In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import interpolate
import datetime
import plotly.express as px

In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    days_in_year = (decimal_year - year) * 365
    date = pd.to_datetime(year, format='%Y') + pd.to_timedelta(days_in_year, unit='D')
    return date

In [ ]:
data = pd.read_csv('samples_20251031.csv')
data['path'] = data['sample'].str[0]
data['sample1'] = pd.to_numeric(data['sample'].str[2:-1])
data

In [ ]:
dark_start = 7/12 # which month the dark stripes start. For example for solomon april may.

paths = pd.unique(data['path'])
res_array = np.array([])
res_array2 = np.array([])

for p in paths:

    data_iter = data[(data['path']==p)]
    data_iter_ = data[(data['path']==p) & (~data['age1'].isna())]
    f  = interpolate.interp1d(data_iter_['sample1'], data_iter_['age1']+dark_start, bounds_error=False)
    res = f(data_iter['sample1'].values)
    res_array = np.append(res_array,res)

data['age_lin'] = res_array
data

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 5))

ax.plot(data['sample1'], data['age_lin'], ".")
ax.plot(data['sample1'], data['age1'], "x")


In [ ]:
data.to_csv("interp_20251031.csv")

In [ ]:
interp  = pd.read_excel('interp_20251029.xlsx')
interp['datetime'] = decimal_year_to_datetime(interp["absolute_age"])
interp.drop("Unnamed: 0",axis=1, inplace=True)
interp.rename(columns={interp.columns[6]:"d13C", interp.columns[7]:"d16O"}, inplace=True)
interp.set_index("datetime",inplace=True)
resampled = interp.iloc[:,6:].resample('4MS').mean()
resampled.to_csv('resampled.csv')
resampled

In [ ]:
interp.rename(columns={interp.columns[6]:"d13C", interp.columns[7]:"d16O"})


In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 5))

ax.plot(interp.iloc[:,6], ".")
ax.plot(resampled.iloc[:,0])
ax.set_title(r'$\delta 13 C$')

ax.grid()
#fig.savefig('d13c.pdf')

In [ ]:
resampled['d13C']

In [ ]:
fig = px.line(resampled, x=resampled.index,  y="d13C")
fig.show()

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 5))

ax.plot(interp.iloc[:,7], ".")
ax.plot(resampled.iloc[:,1])
ax.set_title(r'$\delta 18 O$')
#fig.savefig('d16o.pdf')

In [ ]:
data_luc = pd.read_table('../2025_04_30_all_data/NewPath1926262012.csv', header=None, sep=',')
data_luc['datetime'] = decimal_year_to_datetime(data_luc[0])
data_luc.set_index('datetime', inplace=True)
data_luc

In [ ]:
fig, ax = plt.subplots(1, figsize=(8, 5))

ax.plot(interp.iloc[:,6], ".")
ax.plot(resampled.iloc[:,0],label="memo")

ax.plot(data_luc.iloc[:,1], alpha=0.5, label="Luc")

ax.legend()
ax.set_title(r'$\delta 13 C$')
#fig.savefig('d13c_compare.pdf')




In [ ]:
data_luc